# Preprocesamiento: deltas TARGET-TARGET (aumento de datos)

Este notebook es un pipeline **separado** de `01_preprocesamiento.ipynb` (que sigue igual,
sin tocar). En vez de deltas target-background (target menos background/healthy), acá se
construyen deltas **target-target**: la diferencia entre dos mediciones de target DISTINTAS
que se solapan geométricamente entre sí (mismo líquido, tamaños de target distintos), simulando
crecimiento/reducción/cambio de forma de un stroke.

**No modifica ni sobreescribe** ningún archivo original:
- `target_intersections.csv` (positions/stroke_positions.m -- pares de posiciones/ángulos que
  se solapan geométricamente, calculado SOLO entre tamaños de target distintos: T1×T2, T1×T3, T2×T3)
- `simulation_dataset_V2.h5` (dominio A)
- `measurements_database_robot_V2.h5` (dominio B)

Solo **lee** esos archivos y escribe dos NUEVOS:
- `synthetic_dataset_6x12_targettarget.h5`
- `experimental_dataset_6x12_targettarget.h5`

**Nota importante sobre el dominio sintético**: `Esc` en `simulation_dataset_V2.h5` ya es
`target - background`, con el MISMO background (una sola resolución FEM, reutilizada para las
1216 escenas -- confirmado en `main_generate_simulation_dataset.m`). Por eso el delta
target-target sintético es simplemente `Esc_A - Esc_B` (el background se cancela solo, no hace
falta reconstruir nada).

**Nota importante sobre el dominio experimental**: acá el delta target-target NO puede salir de
restar los `dS` ya diferenciados de `01_preprocesamiento.ipynb` -- ese `dS` usa el 'healthy' MÁS
CERCANO anterior a cada acquisition, que puede ser DISTINTO para el target A y el target B del
par (si hubo una recalibración healthy entre medio). Restar dos `dS` con distinto background
dejaría un residuo. Por eso acá se usa un loader nuevo (`load_experimental_raw_targets`) que lee
el campo TOTAL CRUDO `S_target` de cada acquisition (sin restar ningún background), y el delta
target-target es directamente `S_target_A_raw - S_target_B_raw` -- el background nunca entra en
la cuenta, así que no importa cuál healthy precedió a cada uno.

In [ ]:
import csv

import matplotlib.pyplot as plt
import numpy as np

from data_io import (
    AB_MASK,
    ANGLE_CSV_TO_DATASET,
    compress_ab_pairs_list,
    csv_angle_to_dataset,
    csv_position_to_dataset,
    load_experimental_raw_targets,
    load_synthetic_domain,
    parse_eps_group,
    parse_target_size_id,
    save_compressed_dataset_h5,
)

# --- PATHS (adjust if needed) ---
CSV_PATH = "/Users/felipemedina/Desktop/POLITO/Micriwave Brain/positions/target_intersections.csv"
SIM_H5_PATH = ("/Users/felipemedina/Desktop/POLITO/Micriwave Brain/"
               "OSMI---Open-Source-Microwave-Imaging-main/Examples/"
               "Head_imaging/results/simulation_dataset_V2.h5")
ROBOT_H5_PATH = ("/Users/felipemedina/Desktop/POLITO/Micriwave Brain/"
                 "Robot_Acquisition/measurements_database_robot_V2.h5")
FREQ_HZ = 1.3e9

# Mismas 18 acquisitions corrompidas excluidas en 01_preprocesamiento.ipynb
# (acq_1125 a acq_1142) -- se excluyen tambien aca, tanto si aparecen como
# target_A o target_B de un par.
EXCLUDE_ACQ_NUMBERS = range(1125, 1143)

# Tolerancias para emparejar (posicion, rotacion) transformadas del CSV
# contra las muestras reales del dataset.
POS_TOL_MM = 1.0
ROT_TOL_DEG = 1.0

EPS_GROUPS = ["Hem1", "Hem2", "Isc1", "Isc2"]
TARGET_AXES_MM = {"1": (15, 10), "2": (30, 25), "3": (55, 40)}  # rx, ry por target_type

# --- OUTPUTS (archivos NUEVOS, los originales nunca se tocan) ---
OUT_SYNTHETIC_TT_H5 = "./synthetic_dataset_6x12_targettarget.h5"
OUT_EXPERIMENTAL_TT_H5 = "./experimental_dataset_6x12_targettarget.h5"

## STEP 1 -- Cargar `target_intersections.csv` y transformar a coordenadas del dataset

El CSV está en el marco de coordenadas propio de `positions/stroke_positions.m` (basado en la
imagen del contorno del phantom), NO en el marco `pos_x_mm`/`pos_y_mm`/`rotation_deg` que usan
los `.h5` del dataset. Se aplican dos transformaciones, ambas **verificadas empíricamente** (ver
docstrings de `csv_position_to_dataset`/`csv_angle_to_dataset` en `data_io.py`):

- **Posición**: intercambio puro de ejes `(x_dataset, y_dataset) = (y_csv, x_csv)`. Verificado
  comparando las 42+26+8 posiciones válidas de `valid_positions_T1/T2/T3.csv` contra las
  posiciones reales usadas en `simulation_dataset_V2.h5`: de 8 transformaciones candidatas, esta
  es la única con 100% de coincidencia en los tres tamaños de target simultáneamente.
- **Ángulo**: como el intercambio de ejes es una REFLEXIÓN (no una rotación), el ángulo de la
  elipse también cambia: `ang_dataset = 90 - ang_csv` (mod 180°). Derivado algebraicamente
  representando el contorno de la elipse y aplicando la reflexión; el resultado es una
  permutación cerrada y auto-consistente dentro del mismo conjunto {-45,0,45,90}
  (`-45->-45, 0->90, 45->45, 90->0`), fuerte indicio de que la fórmula es correcta.

In [ ]:
with open(CSV_PATH) as f:
    csv_rows_raw = list(csv.DictReader(f))

print(f"Filas leidas de target_intersections.csv: {len(csv_rows_raw)}")
print("Ejemplo (crudo, marco 'positions'):", csv_rows_raw[0])

pairs_transformed = []
for row in csv_rows_raw:
    xA, yA = csv_position_to_dataset(float(row["xA_mm"]), float(row["yA_mm"]))
    xB, yB = csv_position_to_dataset(float(row["xB_mm"]), float(row["yB_mm"]))
    angA = csv_angle_to_dataset(float(row["ang_A_deg"]))
    angB = csv_angle_to_dataset(float(row["ang_B_deg"]))
    pairs_transformed.append({
        "target_A": row["target_A"], "target_B": row["target_B"],
        "xA": xA, "yA": yA, "angA": angA,
        "xB": xB, "yB": yB, "angB": angB,
    })

print("Ejemplo (transformado, marco 'dataset'):", pairs_transformed[0])

## STEP 1b -- Verificación visual: ¿los pares siguen solapando tras la transformación?

Chequeo de sanidad barato pero contundente: si la transformación de posición/ángulo estuviera
mal, los pares dejarían de solapar geométricamente en el marco del dataset (la transformación es
una isometría -- reflexión + reflexión consistente del ángulo -- así que si está bien, el
solapamiento se preserva exactamente). Se reimplementa `ellipses_intersect` de
`stroke_positions.m` en Python y se corre sobre las 10274 filas transformadas.

In [ ]:
def ellipse_boundary(rx, ry, ang_deg, cx, cy, n=60):
    t = np.linspace(0, 2 * np.pi, n)
    a = np.deg2rad(ang_deg)
    xe, ye = rx * np.cos(t), ry * np.sin(t)
    xr = xe * np.cos(a) - ye * np.sin(a) + cx
    yr = xe * np.sin(a) + ye * np.cos(a) + cy
    return xr, yr


def point_in_polygon(px, py, poly_x, poly_y):
    n = len(poly_x)
    inside = False
    j = n - 1
    for i in range(n):
        xi, yi = poly_x[i], poly_y[i]
        xj, yj = poly_x[j], poly_y[j]
        if ((yi > py) != (yj > py)) and (px < (xj - xi) * (py - yi) / (yj - yi + 1e-300) + xi):
            inside = not inside
        j = i
    return inside


def ellipses_intersect(rx1, ry1, ang1, cx1, cy1, rx2, ry2, ang2, cx2, cy2):
    xb1, yb1 = ellipse_boundary(rx1, ry1, ang1, cx1, cy1)
    xb2, yb2 = ellipse_boundary(rx2, ry2, ang2, cx2, cy2)
    for px, py in zip(xb1, yb1):
        if point_in_polygon(px, py, xb2, yb2):
            return True
    for px, py in zip(xb2, yb2):
        if point_in_polygon(px, py, xb1, yb1):
            return True
    return (point_in_polygon(cx1, cy1, xb2, yb2) or
            point_in_polygon(cx2, cy2, xb1, yb1))


n_ok, n_fail = 0, 0
fails = []
for i, row in enumerate(pairs_transformed):
    rxA, ryA = TARGET_AXES_MM[row["target_A"].replace("T", "")]
    rxB, ryB = TARGET_AXES_MM[row["target_B"].replace("T", "")]
    ok = ellipses_intersect(rxA, ryA, row["angA"], row["xA"], row["yA"],
                             rxB, ryB, row["angB"], row["xB"], row["yB"])
    if ok:
        n_ok += 1
    else:
        n_fail += 1
        if len(fails) < 5:
            fails.append((i, row))

print(f"Verificacion de solapamiento tras transformar: {n_ok}/{len(pairs_transformed)} OK, {n_fail} fallos")
if fails:
    print("Ejemplos de fallo:", fails)

In [ ]:
# Visualizacion de un ejemplo concreto: las dos elipses transformadas, en el
# marco del dataset, deberian verse solapando.
example = pairs_transformed[0]
rxA, ryA = TARGET_AXES_MM[example["target_A"].replace("T", "")]
rxB, ryB = TARGET_AXES_MM[example["target_B"].replace("T", "")]

xbA, ybA = ellipse_boundary(rxA, ryA, example["angA"], example["xA"], example["yA"], n=200)
xbB, ybB = ellipse_boundary(rxB, ryB, example["angB"], example["xB"], example["yB"], n=200)

fig, ax = plt.subplots(figsize=(6, 6))
ax.fill(xbA, ybA, alpha=0.4, label=f"{example['target_A']} @ ({example['xA']:.0f},{example['yA']:.0f}) {example['angA']:.0f}deg")
ax.fill(xbB, ybB, alpha=0.4, label=f"{example['target_B']} @ ({example['xB']:.0f},{example['yB']:.0f}) {example['angB']:.0f}deg")
ax.set_aspect("equal")
ax.set_xlabel("pos_x_mm (marco dataset)")
ax.set_ylabel("pos_y_mm (marco dataset)")
ax.set_title("Ejemplo de par target-target transformado (deberian solapar)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## STEP 2 -- Dominio sintético: `Esc_A - Esc_B`

Se carga `simulation_dataset_V2.h5` completo (1216 escenas = 3 tamaños × posiciones × 4
rotaciones × 4 líquidos, grid combinatorio COMPLETO), se arma un índice por
`(tamaño, liquido, posicion, rotacion)`, y para cada fila transformada del CSV × cada uno de los
4 grupos de líquido, se buscan las dos muestras (A y B) y se resta directamente `Esc_A - Esc_B`
(sin tocar el `.h5` crudo del FEM, sin reconstruir ningún background).

In [ ]:
A_list, A_meta = load_synthetic_domain(SIM_H5_PATH)


def build_index(meta_list, pos_tol_mm=POS_TOL_MM, rot_tol_deg=ROT_TOL_DEG):
    """(tamano, grupo_liquido, pos_x, pos_y, rotacion) -> primer indice que matchea."""
    idx = {}
    for i, m in enumerate(meta_list):
        size_id = parse_target_size_id(m.get("target_type"))
        eps_group = parse_eps_group(m.get("target_eps_label"))
        if size_id is None or eps_group is None:
            continue
        px = round(m.get("pos_x_mm", np.nan) / pos_tol_mm) * pos_tol_mm
        py = round(m.get("pos_y_mm", np.nan) / pos_tol_mm) * pos_tol_mm
        rot = round(m.get("rotation_deg", np.nan) / rot_tol_deg) * rot_tol_deg
        key = (size_id, eps_group, px, py, rot)
        if key not in idx:
            idx[key] = i
    return idx


def build_pair_key(target_label, x, y, ang):
    size_id = target_label.replace("T", "")
    px = round(x / POS_TOL_MM) * POS_TOL_MM
    py = round(y / POS_TOL_MM) * POS_TOL_MM
    rot = round(ang / ROT_TOL_DEG) * ROT_TOL_DEG
    return size_id, px, py, rot


A_index = build_index(A_meta)
print(f"Indice dominio A: {len(A_index)} combos unicos de {len(A_meta)} muestras totales")

tt_synth_deltas, tt_synth_meta = [], []
n_found, n_missing = 0, 0
for row in pairs_transformed:
    size_A, pxA, pyA, rotA = build_pair_key(row["target_A"], row["xA"], row["yA"], row["angA"])
    size_B, pxB, pyB, rotB = build_pair_key(row["target_B"], row["xB"], row["yB"], row["angB"])
    for eps in EPS_GROUPS:
        iA = A_index.get((size_A, eps, pxA, pyA, rotA))
        iB = A_index.get((size_B, eps, pxB, pyB, rotB))
        if iA is None or iB is None:
            n_missing += 1
            continue
        n_found += 1
        tt_synth_deltas.append(A_list[iA] - A_list[iB])
        tt_synth_meta.append({
            "target_type_A": A_meta[iA]["target_type"], "target_type_B": A_meta[iB]["target_type"],
            "pos_x_A_mm": A_meta[iA]["pos_x_mm"], "pos_y_A_mm": A_meta[iA]["pos_y_mm"],
            "rotation_A_deg": A_meta[iA]["rotation_deg"],
            "pos_x_B_mm": A_meta[iB]["pos_x_mm"], "pos_y_B_mm": A_meta[iB]["pos_y_mm"],
            "rotation_B_deg": A_meta[iB]["rotation_deg"],
            "target_eps_label": A_meta[iA]["target_eps_label"],
        })

print(f"Sintetico: {n_found} pares target-target construidos, {n_missing} combos sin match "
      f"(de {len(pairs_transformed)} filas x 4 liquidos = {len(pairs_transformed)*4} intentos)")

In [ ]:
# Visualizacion de un ejemplo: Esc_A, Esc_B, y su delta target-target.
def show_matrix(mat, title=""):
    fig, ax = plt.subplots(1, 2, figsize=(9, 4))
    im0 = ax[0].matshow(np.real(mat))
    ax[0].set_title(f"{title} - Re")
    plt.colorbar(im0, ax=ax[0])
    im1 = ax[1].matshow(np.imag(mat))
    ax[1].set_title(f"{title} - Im")
    plt.colorbar(im1, ax=ax[1])
    fig.tight_layout()
    plt.show()


idx_show = 0
m = tt_synth_meta[idx_show]
print(f"Ejemplo: {m['target_type_A']} @ ({m['pos_x_A_mm']:.0f},{m['pos_y_A_mm']:.0f}) "
      f"{m['rotation_A_deg']:.0f}deg  MENOS  {m['target_type_B']} @ "
      f"({m['pos_x_B_mm']:.0f},{m['pos_y_B_mm']:.0f}) {m['rotation_B_deg']:.0f}deg "
      f"-- liquido {m['target_eps_label']}")
show_matrix(tt_synth_deltas[idx_show], title="Delta target-target sintetico (ejemplo)")

## STEP 3 -- Dominio experimental: `S_target_A_raw - S_target_B_raw`

`load_experimental_raw_targets` devuelve el campo TOTAL crudo de cada acquisition `target`
(nunca resta background), excluyendo las mismas 18 acquisitions corrompidas de
`01_preprocesamiento.ipynb`. A diferencia del dominio sintético (grid combinatorio completo), acá
NO todos los combos posición×rotación×líquido tienen una medición real -- se usan solo los que
existen.

In [ ]:
S_list, S_meta = load_experimental_raw_targets(
    ROBOT_H5_PATH, freq_hz=FREQ_HZ, exclude_acq_numbers=EXCLUDE_ACQ_NUMBERS)

B_index = build_index(S_meta)
print(f"Indice dominio B: {len(B_index)} combos unicos de {len(S_meta)} muestras totales")

tt_exp_deltas, tt_exp_meta = [], []
n_found_exp, n_missing_exp = 0, 0
for row in pairs_transformed:
    size_A, pxA, pyA, rotA = build_pair_key(row["target_A"], row["xA"], row["yA"], row["angA"])
    size_B, pxB, pyB, rotB = build_pair_key(row["target_B"], row["xB"], row["yB"], row["angB"])
    for eps in EPS_GROUPS:
        iA = B_index.get((size_A, eps, pxA, pyA, rotA))
        iB = B_index.get((size_B, eps, pxB, pyB, rotB))
        if iA is None or iB is None:
            n_missing_exp += 1
            continue
        n_found_exp += 1
        tt_exp_deltas.append(S_list[iA] - S_list[iB])
        tt_exp_meta.append({
            "target_type_A": S_meta[iA]["target_type"], "target_type_B": S_meta[iB]["target_type"],
            "pos_x_A_mm": S_meta[iA]["pos_x_mm"], "pos_y_A_mm": S_meta[iA]["pos_y_mm"],
            "rotation_A_deg": S_meta[iA]["rotation_deg"],
            "pos_x_B_mm": S_meta[iB]["pos_x_mm"], "pos_y_B_mm": S_meta[iB]["pos_y_mm"],
            "rotation_B_deg": S_meta[iB]["rotation_deg"],
            "target_eps_label": S_meta[iA]["target_eps_label"],
            "acq_num_A": S_meta[iA]["acq_num"], "acq_num_B": S_meta[iB]["acq_num"],
        })

print(f"Experimental: {n_found_exp} pares target-target construidos, {n_missing_exp} combos sin match")

In [ ]:
idx_show = 0
m = tt_exp_meta[idx_show]
print(f"Ejemplo: acq_{m['acq_num_A']} ({m['target_type_A']} @ "
      f"({m['pos_x_A_mm']:.0f},{m['pos_y_A_mm']:.0f}) {m['rotation_A_deg']:.0f}deg)  MENOS  "
      f"acq_{m['acq_num_B']} ({m['target_type_B']} @ "
      f"({m['pos_x_B_mm']:.0f},{m['pos_y_B_mm']:.0f}) {m['rotation_B_deg']:.0f}deg) -- "
      f"liquido {m['target_eps_label']}")
show_matrix(tt_exp_deltas[idx_show], title="Delta target-target experimental (ejemplo)")

## STEP 4 -- Comprimir a 6x12 y guardar

Mismo formato que `01_preprocesamiento.ipynb` (`compress_ab_pairs_list` + `save_compressed_dataset_h5`),
así los archivos nuevos son compatibles con `load_compressed_dataset_h5`/`load_compressed_datasets`
y se pueden usar directamente para entrenar un Cycle-GAN target-target o combinado, sin cambios en
el pipeline de entrenamiento.

In [ ]:
tt_synth_compressed = compress_ab_pairs_list(tt_synth_deltas)
tt_exp_compressed = compress_ab_pairs_list(tt_exp_deltas)

save_compressed_dataset_h5(
    OUT_SYNTHETIC_TT_H5, tt_synth_compressed, tt_synth_meta,
    freq_hz=FREQ_HZ, domain_name="synthetic_target_target",
    source_path=f"{SIM_H5_PATH} + {CSV_PATH}")

save_compressed_dataset_h5(
    OUT_EXPERIMENTAL_TT_H5, tt_exp_compressed, tt_exp_meta,
    freq_hz=FREQ_HZ, domain_name="experimental_target_target",
    source_path=f"{ROBOT_H5_PATH} + {CSV_PATH}")

## STEP 5 -- Verificacion final

Recarga los dos archivos nuevos y confirma cantidades/formas.

In [ ]:
from data_io import load_compressed_dataset_h5

A_tt_check, A_tt_meta_check, freq_A_tt = load_compressed_dataset_h5(OUT_SYNTHETIC_TT_H5)
B_tt_check, B_tt_meta_check, freq_B_tt = load_compressed_dataset_h5(OUT_EXPERIMENTAL_TT_H5)

print("=== Resumen final ===")
print(f"synthetic_dataset_6x12_targettarget.h5    : {len(A_tt_check)} muestras, "
      f"forma por muestra {A_tt_check[0].shape}, freq={freq_A_tt/1e9:.3f} GHz")
print(f"experimental_dataset_6x12_targettarget.h5 : {len(B_tt_check)} muestras, "
      f"forma por muestra {B_tt_check[0].shape}, freq={freq_B_tt/1e9:.3f} GHz")

same_size_synth = sum(1 for m in A_tt_meta_check if m["target_type_A"] == m["target_type_B"])
same_size_exp = sum(1 for m in B_tt_meta_check if m["target_type_A"] == m["target_type_B"])
print(f"\nPares con mismo target_type en A y B (deberia ser 0 -- el CSV solo tiene pares de "
      f"tamanos distintos): sintetico={same_size_synth}, experimental={same_size_exp}")